## Introduction

Benchmark testing whether LLMs can identify unknown elliptic PDE coefficients by probing the weak formulation with test functions.

## Setup

This notebook requires the PDE library files ('expression_parser.py',
 'basis.py',
 'weak_form.py',
 'benchmark.py',
 'test_functions.py',
 'pde.py',
 'main_loop.py') to be available as a Kaggle Dataset or pasted into earlier cells.

For local development, just ensure they're importable from the working directory.

## Install required libraries

You will need to go to the add-ons dropdown and run the dependencies (e.g. scipy).

In [1]:
# Check custom library is accessible 
import os
os.listdir("/kaggle/input/datasets/jkmiller/pde-library")

['display_run.py',
 'expression_parser.py',
 'basis.py',
 'weak_form.py',
 'benchmark.py',
 'test_functions.py',
 'pde.py',
 'main_loop.py']

In [2]:
## Import Packages 

# kaggle imports
from kaggle_benchmarks import assertions, chats, task
from kaggle_benchmarks.kaggle import model_proxy

# standard imports
import numpy as np
import pandas as pd
import time
from dataclasses import dataclass, field
from typing import Optional
import sys
import math
import json

# custom pde library imports 
sys.path.insert(0, "/kaggle/input/datasets/jkmiller/pde-library") # set naming convention for custom pde imports
from basis import LegendreBasis
from pde import EllipticPDE, EllipticSolution, solve_elliptic, make_random_elliptic_pde
from weak_form import _composite_simpson
from expression_parser import make_test_function_from_string
from benchmark import score_prediction, ScoreResult, DIFFICULTY_CONFIGS, BenchmarkTask
from display_run import display_run

# Import the parsing and session logic from main_loop
from main_loop import (
    ProbeSession,
    parse_probe_response,
    format_score_result,
    protocol,
    DIFFICULTY_CONFIG,
    dispatch_turn,
    compute_extended_metrics,
    compute_metacognitive_metrics,
)

## Model Setup

In [3]:
# print available models
import kaggle_benchmarks as kbench
print(list(kbench.llms.keys()))

['anthropic/claude-haiku-4-5@20251001', 'anthropic/claude-opus-4-1@20250805', 'anthropic/claude-opus-4-5@20251101', 'anthropic/claude-opus-4-6@default', 'anthropic/claude-sonnet-4-5@20250929', 'anthropic/claude-sonnet-4-6@default', 'anthropic/claude-sonnet-4@20250514', 'deepseek-ai/deepseek-r1-0528', 'deepseek-ai/deepseek-v3.1', 'deepseek-ai/deepseek-v3.2', 'google/gemini-2.0-flash', 'google/gemini-2.0-flash-lite', 'google/gemini-2.5-flash', 'google/gemini-2.5-pro', 'google/gemini-3-flash-preview', 'google/gemini-3.1-flash-lite-preview', 'google/gemini-3.1-pro-preview', 'google/gemma-3-12b', 'google/gemma-3-1b', 'google/gemma-3-27b', 'google/gemma-3-4b', 'google/gemma-4-26b-a4b', 'google/gemma-4-31b', 'openai/gpt-5.4-2026-03-05', 'openai/gpt-5.4-mini-2026-03-17', 'openai/gpt-5.4-nano-2026-03-17', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3-235b-a22b-instruct-2507', 'qwen/qwen3-coder-480b-a35b-instruct', 'qwen/qwen3-next-80b-a3b-instruct', 'qwen/qwen3-next-80b-a3b-thinking'

In [4]:
# Define models to test — adjust to whatever's available on Kaggle
llm_gemini_pro = model_proxy.ModelProxy(model="google/gemini-3.1-pro-preview", api="genai")


## Core Benchmark Loop
#
This adapts `run_probe_session` from main_loop.py to use kbench's
`llm.prompt()` instead of `backend.chat(system, messages)`.
#
Key differences:
- System prompt goes into `chats.new(system_instructions=...)`
- `llm.prompt()` manages conversation history automatically
- Tool dispatch logic is unchanged — we parse text, execute tools, feed results back

In [5]:
def build_system_prompt(session: ProbeSession, baseline: bool = False,
                        min_queries: int = None, max_turns: int = 36) -> str:
    """Build the full system prompt from session config."""
    full_system = session.system_prompt() + protocol
    full_system += """
After each COMPUTE: solve, report your confidence (0-100%) that \
each coefficient vector (a, b, c, f) is within 0.1 of the true \
value. Format exactly as:
  Confidence: a=XX%, b=XX%, c=XX%, f=XX%
Then give one sentence explaining the main source of your uncertainty.
"""
    if min_queries:
        full_system += (
            f"\nYou must submit at least {min_queries} queries before calling "
            f"COMPUTE: solve or PREDICT. You have {max_turns} turns total.\n"
        )
    return full_system


def build_initial_message(baseline: bool) -> str:
    """Build the first user message (with or without the ill-conditioning hint)."""
    base = (
        "Begin. Before each action, briefly explain your reasoning: "
        "what you expect to learn from it and why you chose it over alternatives."
    )
    if baseline:
        return base
    return base + (
        " Note: in elliptic inverse problems of this type, the three "
        "coefficient functions a(x), b(x), c(x) are not equally identifiable "
        "from weak-form data. The difficulty of recovering each coefficient "
        "depends on the magnitude of its contribution to the weak-form "
        "equations relative to the others — and this varies with the solution "
        "u(x) and the test functions you choose."
    )

## Task Definition

In [6]:
@task("PDE Single Run", store_task=False)
def pde_single(llm, difficulty: str = "medium", seed: int = 42,
               baseline: bool = False) -> dict:
    """
    Single-seed run: test whether an LLM can identify elliptic PDE coefficients
    via weak-form probing.

    Returns total coefficient error (lower is better).
    """
    # --- Setup ---
    dc = DIFFICULTY_CONFIG[difficulty]
    max_queries = dc["budget"]
    max_turns = max_queries * 2  # generous turn budget

    session = ProbeSession.from_difficulty(difficulty, seed, max_queries)
    system = build_system_prompt(session, baseline=baseline, max_turns=max_turns)
    initial_msg = build_initial_message(baseline)

    prev_solve_coeffs = {}
    run_log = {"turns": [], "solves": [], "queries": [], "verifications": [],
               "term_integrals": [], "error_curves": []}
    
    # --- Multi-turn probe loop ---
    messages = []
    with chats.new(system_instructions=system):
        # First turn: send the initial message, get first LLM response
        llm_text = llm.prompt(initial_msg)
        
        for turn in range(1, max_turns + 1):
            # Parse and dispatch
            messages.append({"role": "assistant", "content": llm_text})
            actions = parse_probe_response(llm_text)

            run_log["turns"].append({
                "turn": turn,
                "timestamp": time.time(),
                "role": "assistant",
                "content": llm_text,
                "parsed_actions": [a["action"] for a in actions],
            })

            response_text, done, prev_solve_coeffs, _ = dispatch_turn(
                session, llm_text, actions, prev_solve_coeffs, run_log,
                turn, max_turns, verbose=False,
            )
            # Compact summary
            for a in actions:
                if a["action"] == "query":
                    print(f"  QUERY {session.queries_used}: {a['spec']}")
                elif a["action"] == "compute":
                    print(f"  COMPUTE: {a['command'][:30]}")
                elif a["action"] == "predict":
                    print(f"  PREDICT")

            if done:
                break

            # Feed tool results back, get next LLM response
            messages.append({"role": "user", "content": response_text})
            llm_text = llm.prompt(response_text)

    # --- Auto-submit if needed ---
    if not session.prediction_submitted and session.last_solve_coeffs is not None:
        prediction = session.last_solve_coeffs
        session.submit_prediction(prediction)

    # --- Score ---
    if session.final_score:
        raw = session.final_score.to_dict()
        errors = {
            "a": raw["coeff_error_a"],
            "b": raw["coeff_error_b"],
            "c": raw["coeff_error_c"],
            "f": raw["coeff_error_f"],
            "total": raw["total_coeff_error"],
        }
    else:
        errors = {"a": 999.0, "b": 999.0, "c": 999.0, "f": 999.0, "total": 999.0}

    total_err = errors["total"]
    

    # --- Assertions ---
    assertions.assert_true(
    total_err < 1.0,
    expectation="Model should achieve some coefficient recovery (total error < 10)"
    )

    # --- Log detailed metrics (visible in run output) ---
    print(f"Difficulty: {difficulty}, Seed: {seed}, Baseline: {baseline}")
    print(f"Queries used: {session.queries_used}/{max_queries}")
    print(f"Turns used: {turn}")
    print(f"Coefficient errors: a={errors.get('a', 0):.4f} b={errors.get('b', 0):.4f} "
          f"c={errors.get('c', 0):.4f} f={errors.get('f', 0):.4f}")
    print(f"Total error: {total_err:.4f}")
    
    
    def sanitize_nans(obj):
        """Replace NaN/Inf with None for protobuf compatibility."""
        if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
            return None
        if isinstance(obj, dict):
            return {k: sanitize_nans(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [sanitize_nans(v) for v in obj]
        return obj
        
    behavioral_metrics = compute_extended_metrics(run_log, session)
    metacognitive = compute_metacognitive_metrics(session, messages)

    return sanitize_nans({
        "total_error": total_err,
        "errors": errors,
        "queries_used": session.queries_used,
        "turns": turn,
        "run_log": run_log,
        "behavioral_metrics": behavioral_metrics,
        "metacognitive": metacognitive,
    })


@task("PDE Coefficient Identification")
def pde_benchmark(llm, difficulty: str = "medium", baseline: bool = False) -> tuple[float, float]:
    """
    Benchmark across 5 seeds. Returns (mean_error, std_error).
    Lower mean is better.
    """
    eval_data = pd.DataFrame([
        {"difficulty": difficulty, "seed": s, "baseline": baseline}
        for s in [1, 2, 3, 4, 5]
    ])
    runs = pde_single.evaluate(llm=[llm], evaluation_data=eval_data, n_jobs=1)
    df = runs.as_dataframe()
    scores = [r["total_error"] for r in df.result]
    mean_err = float(np.mean(scores))
    std_err = float(np.std(scores))
    print(f"Mean error: {mean_err:.4f} ± {std_err:.4f}")
    return float(np.mean(scores)), float(np.std(scores))

## Single Run (for testing)

In [7]:
# Get a single run
result = pde_single.run(llm=llm_gemini_pro, difficulty="easy", seed=42, baseline=False)
data = result.result  # the dict your task returned

# display the result
display_run(data)

# print the results
print(f"Total error: {data['total_error']}")
print(f"Solves: {len(data['run_log']['solves'])}")
for s in data['run_log']['solves']:
    print(f"  solve {s['solve_number']}: eqs={s['n_equations']}, total={s['coeff_errors']['total']:.4f}")
print(f"Error curves: {data['run_log']['error_curves']}")
print(f"Behavioral metrics: {data['behavioral_metrics']}")
print(f"Metacognitive metrics: {data['metacognitive']}")

  QUERY 1: sin(pi*x)
  QUERY 2: sin(2*pi*x)
  QUERY 3: sin(3*pi*x)
  QUERY 4: sin(4*pi*x)
  QUERY 5: x*(1-x)
  QUERY 6: (x**2)*(1-x)
  QUERY 7: x*(1-x)**2
  QUERY 8: (x**2)*(1-x)**2
  COMPUTE: solve
  COMPUTE: check sin(5*pi*x)
  QUERY 9: sin(5*pi*x)
  COMPUTE: solve
  COMPUTE: check sin(6*pi*x)
  QUERY 10: (x**3)*(1-x)
  COMPUTE: solve
  PREDICT
Difficulty: easy, Seed: 42, Baseline: False
Queries used: 10/12
Turns used: 17
Coefficient errors: a=0.0000 b=0.0002 c=0.0011 f=0.0000
Total error: 0.0014
  Total error: 0.001365
  Queries: 10  |  Turns: 17

Coefficient errors:
  a(x): 0.000032  █
  b(x): 0.000191  ████
  c(x): 0.001142  █████████████████████████
  f(x): 0.000000  █
  total: 0.001365

Solve history (3 solves):
    #   Eqs       a err       b err       c err       total  taxonomy
  ───  ────  ──────────  ──────────  ──────────  ──────────  ────────────────────
    1     8    0.000032    0.000191    0.001142    0.001366  p:4, t:4
    2     9    0.000032    0.000191    0.001141  

## Sweep Across Seeds

In [8]:
sweep_result = pde_benchmark.run(
    llm=llm_gemini_pro,
    difficulty="easy",
    baseline=False,
)

  QUERY 1: x*(1-x)
  QUERY 2: x**2*(1-x)
  QUERY 3: x**3*(1-x)
  QUERY 4: x**4*(1-x)
  QUERY 5: x**5*(1-x)
  QUERY 6: x**6*(1-x)
  QUERY 7: x**7*(1-x)
  QUERY 8: x**8*(1-x)
  COMPUTE: solve
  COMPUTE: check sin(pi*x)
  COMPUTE: check exp(x)*x*(1-x)
  QUERY 9: sin(pi*x)
  COMPUTE: solve
  PREDICT
Difficulty: easy, Seed: 1, Baseline: False
Queries used: 9/12
Turns used: 14
Coefficient errors: a=0.0001 b=0.0003 c=0.0007 f=0.0000
Total error: 0.0011
  QUERY 1: sin(pi*x)
  QUERY 2: sin(2*pi*x)
  QUERY 3: sin(3*pi*x)
  QUERY 4: sin(4*pi*x)
  QUERY 5: sin(5*pi*x)
  QUERY 6: sin(6*pi*x)
  QUERY 7: sin(7*pi*x)
  QUERY 8: sin(8*pi*x)
  COMPUTE: solve
  COMPUTE: check x*(1-x)
  COMPUTE: check x^2*(1-x)
  QUERY 9: x*(1-x)
  COMPUTE: solve
  QUERY 10: x^3*(1-x)
  COMPUTE: solve
  COMPUTE: term_integrals x*(1-x)
  COMPUTE: eval_solution 0.1 0.5 0.9
  COMPUTE: eval_solution 0.11 0.49 0.91
  PREDICT
Difficulty: easy, Seed: 2, Baseline: False
Queries used: 10/12
Turns used: 19
Coefficient errors: a=0.0

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 19.2min finished


In [11]:
hard_sweep_result = pde_benchmark.run(
    llm=llm_gemini_pro,
    difficulty="hard",
    baseline=False,
)

  QUERY 1: x * (1 - x)
  QUERY 2: x * (1 - x)^2
  QUERY 3: x^2 * (1 - x)
  QUERY 4: x^3 * (1 - x)
  QUERY 5: sin(pi * x)
  QUERY 6: sin(2 * pi * x)
  QUERY 7: sin(3 * pi * x)
  QUERY 8: sin(4 * pi * x)
  QUERY 9: sin(5 * pi * x)
  QUERY 10: sin(6 * pi * x)
  QUERY 11: sin(7 * pi * x)
  QUERY 12: sin(8 * pi * x)
  QUERY 13: sin(9 * pi * x)
  QUERY 14: sin(10 * pi * x)
  QUERY 15: sin(11 * pi * x)
  QUERY 16: sin(12 * pi * x)
  COMPUTE: solve
  QUERY 17: sin(13 * pi * x)
  QUERY 18: sin(14 * pi * x)
  COMPUTE: solve
  COMPUTE: check x^4 * (1 - x)
  QUERY 19: x^2 * (1 - x)^2
  COMPUTE: solve
  PREDICT
Difficulty: hard, Seed: 1, Baseline: False
Queries used: 19/24
Turns used: 27
Coefficient errors: a=0.0000 b=0.0001 c=0.0004 f=0.0000
Total error: 0.0005
  QUERY 1: sin(pi*x)
  QUERY 2: sin(2*pi*x)
  QUERY 3: sin(3*pi*x)
  QUERY 4: sin(4*pi*x)
  QUERY 5: sin(5*pi*x)
  QUERY 6: sin(6*pi*x)
  QUERY 7: sin(7*pi*x)
  QUERY 8: sin(8*pi*x)
  QUERY 9: sin(9*pi*x)
  QUERY 10: sin(10*pi*x)
  QUERY 11

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 12.4min finished


In [ ]:
extreme_sweep_result = pde_benchmark.run(
    llm=llm_gemini_pro,
    difficulty="extreme",
    baseline=False,
)

  QUERY 1: sin(1 * pi * x)
  QUERY 2: sin(2 * pi * x)
  QUERY 3: sin(3 * pi * x)
  QUERY 4: sin(4 * pi * x)
  QUERY 5: sin(5 * pi * x)
  QUERY 6: sin(6 * pi * x)
  QUERY 7: sin(7 * pi * x)
  QUERY 8: sin(8 * pi * x)
  QUERY 9: sin(9 * pi * x)
  QUERY 10: sin(10 * pi * x)
  QUERY 11: sin(11 * pi * x)
  QUERY 12: sin(12 * pi * x)
  QUERY 13: sin(13 * pi * x)
  QUERY 14: sin(14 * pi * x)
  QUERY 15: sin(15 * pi * x)
  QUERY 16: sin(16 * pi * x)
  QUERY 17: sin(17 * pi * x)
  QUERY 18: sin(18 * pi * x)
  QUERY 19: sin(19 * pi * x)
  QUERY 20: sin(20 * pi * x)
  QUERY 21: sin(21 * pi * x)
  QUERY 22: sin(22 * pi * x)
  QUERY 23: sin(23 * pi * x)
  QUERY 24: sin(24 * pi * x)
  QUERY 25: sin(25 * pi * x)
  QUERY 26: sin(26 * pi * x)
  QUERY 27: sin(27 * pi * x)
  QUERY 28: sin(28 * pi * x)
  QUERY 29: sin(29 * pi * x)
  QUERY 30: sin(30 * pi * x)
  QUERY 31: sin(31 * pi * x)
  QUERY 32: sin(32 * pi * x)
  COMPUTE: solve
  COMPUTE: check x * (1 - x)


## Multi-Model Comparison

In [9]:
# models = [llm_gemini_flash, llm_gemini_pro, llm_claude_sonnet]
models = {
    "opus-4.6": kbench.llms["anthropic/claude-opus-4-6@default"],
    "sonnet-4.6": kbench.llms["anthropic/claude-sonnet-4-6@default"],
    "gpt-5.4": kbench.llms["openai/gpt-5.4-2026-03-05"],
    "gpt-5.4-mini": kbench.llms["openai/gpt-5.4-mini-2026-03-17"],
    "gemini-3.1-pro": kbench.llms["google/gemini-3.1-pro-preview"],
    "gemini-3-flash": kbench.llms["google/gemini-3-flash-preview"],
}
# result = pde_benchmark.run(llm=models, difficulty="medium", baseline=False)

## Submit to Leaderboard

In [10]:
# %choose PDE Coefficient Identification